In [1]:
import os
import sys
import pandas as pd
import numpy as np

import json
import pickle

In [2]:
with open("/projects/extern/nhr/nhr_ni/nib00021/dir.project/hamza/repos/bbox_detection_sam3/ssv2_full_bbox_analysis/ssv2_full_bbox_dataframes.pkl", 'rb') as f:
    data = pickle.load(f)
    df_images = data['df_images']

print(df_images.shape)

(193690, 5)


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

In [4]:
df_images.head(2)

,image_id,height,width,area,num_bboxes
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1


In [ ]:
df_images['id'] = df_images['image_id'].str.split("ssv2_clips_").str[1].str.split('.').str[0]
df_images.head(2)

,image_id,height,width,area,num_bboxes,id
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1,78687
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1,42326


In [11]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/labels/train.json', 'r') as f:
    train_raw = json.load(f)

with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/labels/validation.json', 'r') as f:
    val_raw = json.load(f)

df_train = pd.DataFrame(train_raw)
df_train['split'] = 'train'

df_val = pd.DataFrame(val_raw)
df_val['split'] = 'val'

ssv2_labels_df = pd.concat([df_train, df_val], ignore_index=True)
print(ssv2_labels_df.shape)
ssv2_labels_df.head(2)

(193690, 5)


,id,label,template,placeholders,split
0,78687,holding potato next to vicks vaporub bottle,Holding [something] next to [something],"[potato, vicks vaporub bottle]",train
1,42326,spreading margarine onto bread,Spreading [something] onto [something],"[margarine, bread]",train


In [ ]:
for col in ssv2_labels_df.columns:
    print(f"{col:14}: {ssv2_labels_df[col].astype(str).nunique()}")

id            : 193690
label         : 123858
template      : 174
placeholders  : 65256
split         : 2


In [12]:
df = pd.merge(df_images, ssv2_labels_df[['id','template','split']], how = 'left', on = ['id']) # LEFT JOIN
print(df.shape)
df.head(3)

(193690, 8)


,image_id,height,width,area,num_bboxes,id,template,split
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1,78687,Holding [something] next to [something],train
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1,42326,Spreading [something] onto [something],train
2,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_100904.jpg,240,427,102480,0,100904,Putting [something] on a surface,train


In [ ]:
# If we drop the 0 bbox examples, will it mess up the train and val split percentages? 
# Answer is No
print(df.split.value_counts(1))
print(df[df.num_bboxes>0].split.value_counts(1))

split
train    0.872079
val      0.127921
Name: proportion, dtype: float64
split
train    0.865353
val      0.134647
Name: proportion, dtype: float64


In [16]:
print(df.template.nunique())
print(df[df.num_bboxes>0].template.nunique())

174
174


In [17]:
df = df[df.num_bboxes>0]
df.shape

(150289, 8)

In [ ]:
# Get all unique values of df.template which are action classes in ssv2 dataset, and make a text file with all these action classes, and unique integer id one per line. 
action_classes = df.template.unique()
